In [2]:
import pandas as pd
import re
import os
import numpy as np

In [3]:
df = pd.read_csv('./clean_data/combined_players_data.csv')

C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\274292837.py:1: DtypeWarning: Columns (0: nation_position, 1: nation_logo_url) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./clean_data/combined_players_data.csv')


In [ ]:
def combine_fifa_datasets(base_path="FIFA_PLAYERDATA", output_filename="combined_players_data.csv"):
    """
    Combines FIFA player datasets from players_15.csv to players_22.csv into a single DataFrame.
    Saves the combined DataFrame to a new CSV file.
    """
    all_data = []
    
    print(f"Combining datasets from {base_path}...")

    for year in range(15, 23):  # Includes 15 to 22
        filename = f"players_{year}.csv"
        filepath = os.path.join(base_path, filename)
        
        if os.path.exists(filepath):
            print(f"Reading {filename}...")
            try:
                df = pd.read_csv(filepath)
                all_data.append(df)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
        else:
            print(f"File not found: {filename}")
            
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"Successfully combined {len(all_data)} datasets. Total rows: {len(combined_df)}")
        
        output_filepath = os.path.join(os.getcwd(), output_filename)
        combined_df.to_csv(output_filepath, index=False)
        print(f"Combined data saved to {output_filepath}")
    else:
        print("No data was combined.")


In [16]:
def clean_financial_columns(df, columns):
    """
    Cleans financial columns in a DataFrame by converting string representations
    (e.g., '€100.5M', '€50K') to numeric values (in euros).

    Args:
        df (pd.DataFrame): The DataFrame to process.
        columns (list): A list of column names to clean.

    Returns:
        pd.DataFrame: The DataFrame with cleaned financial columns.
    """
    for col in columns:
        # Skip if the column does not exist
        if col not in df.columns:
            print(f"Column '{col}' not found in DataFrame. Skipping.")
            continue

        # Convert column to string to handle various data types, handle NaNs
        df[col] = df[col].astype(str).str.strip()

        # Replace 'M' with 6 zeros, 'K' with 3 zeros, and remove currency symbols/commas
        df[col] = df[col].replace({'M': 'e6', 'K': 'e3', '€': ''}, regex=True)
        
        # Convert to numeric, coercing errors to NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

In [18]:
financial_cols = ['value_eur', 'wage_eur', 'release_clause_eur']

# Clean the financial columns
cleaned_df = clean_financial_columns(df, financial_cols)

print("Cleaned DataFrame:")
print(cleaned_df[financial_cols])
print("" + "="*30 + "")

Cleaned DataFrame:
          value_eur  wage_eur  release_clause_eur
0       100500000.0  550000.0                 NaN
1        79000000.0  375000.0                 NaN
2        54500000.0  275000.0                 NaN
3        52500000.0  275000.0                 NaN
4        63500000.0  300000.0                 NaN
...             ...       ...                 ...
142074      70000.0    1000.0            114000.0
142075     110000.0     500.0            193000.0
142076     100000.0     500.0            175000.0
142077     110000.0     500.0            239000.0
142078     110000.0     500.0            217000.0

[142079 rows x 3 columns]


In [8]:
def clean_physical_columns(df, columns):
    """
    Cleans physical columns (e.g., 'height_cm', 'weight_kg') in a DataFrame
    by removing units ('cm', 'kg') and converting to integer type.

    Args:
        df (pd.DataFrame): The DataFrame to process.
        columns (list): A list of column names to clean.

    Returns:
        pd.DataFrame: The DataFrame with cleaned physical columns.
    """
    for col in columns:
        if col not in df.columns:
            print(f"Column '{col}' not found in DataFrame. Skipping.")
            continue

        # Convert column to string to handle various data types, then remove suffixes
        # Use .str accessor to apply string methods, handling potential non-string types
        df[col] = df[col].astype(str).str.replace('cm', '', regex=False)
        df[col] = df[col].astype(str).str.replace('kg', '', regex=False)
        df[col] = df[col].astype(str).str.strip() # Remove any leading/trailing whitespace

        # Convert to numeric, coercing errors to NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')

        # Convert to nullable integer type (Int64Dtype) to allow NaN values
        df[col] = df[col].astype(pd.Int64Dtype())

    return df


In [14]:
physical_cols = ['height_cm', 'weight_kg']

# Clean the physical columns
cleaned_df = clean_physical_columns(df, physical_cols)

print("Cleaned DataFrame:")
print(cleaned_df[['height_cm','weight_kg']])
print(cleaned_df[['height_cm','weight_kg']].dtypes)
print("" + "="*30 + "")


Cleaned DataFrame:
        height_cm  weight_kg
0             169         67
1             185         80
2             180         80
3             195         95
4             193         92
...           ...        ...
142074        180         64
142075        175         70
142076        178         72
142077        173         66
142078        167         61

[142079 rows x 2 columns]
height_cm    Int64
weight_kg    Int64
dtype: object


In [4]:
def evaluate_attribute_expression(attribute_str):
    """
    Evaluates a FIFA attribute string (e.g., '75+2', '80') and returns an integer.
    Handles NaN values and coercing errors.
    """
    if pd.isna(attribute_str):
        return np.nan
    
    # Ensure it's a string for processing
    attribute_str = str(attribute_str).strip()

    if '+' in attribute_str:
        try:
            parts = attribute_str.split('+')
            base = int(float(parts[0]))  # Use float to handle potential "75.0+2"
            bonus = int(float(parts[1]))
            return base + bonus
        except ValueError:
            return np.nan
    else:
        try:
            return int(float(attribute_str)) # Handles cases like "80.0"
        except ValueError:
            return np.nan

def clean_attribute_expressions(df, columns):
    """
    Cleans specified columns in a DataFrame that contain FIFA attribute expressions
    (e.g., '75+2') by evaluating them to a single integer.

    Args:
        df (pd.DataFrame): The DataFrame to process.
        columns (list): A list of column names to clean.

    Returns:
        pd.DataFrame: The DataFrame with cleaned attribute columns.
    """
    for col in columns:
        if col not in df.columns:
            print(f"Column '{col}' not found in DataFrame. Skipping.")
            continue
        
        # Apply the evaluation function
        df[col] = df[col].apply(evaluate_attribute_expression)
        
        # Convert to nullable integer type (Int64Dtype) to allow NaN values
        df[col] = df[col].astype(pd.Int64Dtype())
        
    return df

In [5]:
positional_cols = [
            'ls', 'st', 'rs', 'lw', 'lf', 'cf', 'rf', 'rw', 'lam', 'cam', 'ram',
            'lm', 'lcm', 'cm', 'rcm', 'rm', 'lwb', 'ldm', 'cdm', 'rdm', 'rwb',
            'lb', 'lcb', 'cb', 'rcb', 'rb', 'gk'
        ]
cleaned_players_df=clean_attribute_expressions(df,positional_cols)
print(cleaned_players_df[positional_cols].head())
print(cleaned_players_df[positional_cols].dtypes)


   ls  st  rs  lw  lf  cf  rf  rw  lam  cam  ...  ldm  cdm  rdm  rwb  lb  lcb  \
0  92  92  92  95  93  93  93  95   95   95  ...   65   65   65   65  57   48   
1  92  92  92  92  92  92  92  92   92   92  ...   66   66   66   66  60   55   
2  87  87  87  90  90  90  90  90   90   90  ...   67   67   67   67  58   49   
3  90  90  90  87  89  89  89  87   89   89  ...   68   68   68   64  59   58   
4  41  41  41  39  40  40  40  39   39   39  ...   43   43   43   39  39   41   

   cb  rcb  rb  gk  
0  48   48  57  18  
1  55   55  60  19  
2  49   49  58  17  
3  58   58  59  20  
4  41   41  39  90  

[5 rows x 27 columns]
ls     Int64
st     Int64
rs     Int64
lw     Int64
lf     Int64
cf     Int64
rf     Int64
rw     Int64
lam    Int64
cam    Int64
ram    Int64
lm     Int64
lcm    Int64
cm     Int64
rcm    Int64
rm     Int64
lwb    Int64
ldm    Int64
cdm    Int64
rdm    Int64
rwb    Int64
lb     Int64
lcb    Int64
cb     Int64
rcb    Int64
rb     Int64
gk     Int64
dtype: object

In [14]:
print(df.columns.tolist())
print(len(df.columns.tolist()))
arr=['ls', 'st', 'rs', 'lw', 'lf', 'cf', 'rf', 'rw', 'lam', 'cam', 'ram', 'lm', 'lcm', 'cm', 'rcm', 'rm', 'lwb', 'ldm', 'cdm', 'rdm', 'rwb', 'lb', 'lcb', 'cb', 'rcb', 'rb', 'gk']
print(len(arr))

['sofifa_id', 'player_url', 'short_name', 'long_name', 'player_positions', 'overall', 'potential', 'value_eur', 'wage_eur', 'age', 'dob', 'height_cm', 'weight_kg', 'club_team_id', 'club_name', 'league_name', 'league_level', 'club_position', 'club_jersey_number', 'club_loaned_from', 'club_joined', 'club_contract_valid_until', 'nationality_id', 'nationality_name', 'nation_team_id', 'nation_position', 'nation_jersey_number', 'preferred_foot', 'weak_foot', 'skill_moves', 'international_reputation', 'work_rate', 'body_type', 'real_face', 'release_clause_eur', 'player_tags', 'player_traits', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'attacking_crossing', 'attacking_finishing', 'attacking_heading_accuracy', 'attacking_short_passing', 'attacking_volleys', 'skill_dribbling', 'skill_curve', 'skill_fk_accuracy', 'skill_long_passing', 'skill_ball_control', 'movement_acceleration', 'movement_sprint_speed', 'movement_agility', 'movement_reactions', 'movement_balance', 'power

In [22]:
def check_dataframe_inconsistencies(df):
    """
    Scans through each column of a DataFrame and lists potential inconsistencies,
    including NaN values, data types, and unique values for non-numeric columns.

    Args:
        df (pd.DataFrame): The DataFrame to check.

    Returns:
        None: Prints the inconsistencies directly.
    """
    print("--- Checking DataFrame for Inconsistencies ---")
    print(f"Total rows: {len(df)}")
    print(f"Total columns: {len(df.columns)}")

    for col in df.columns:
        print(f"--- Column: '{col}' ---")
        
        # Check for NaN values
        nan_count = df[col].isnull().sum()
        if nan_count > 0:
            nan_percentage = (nan_count / len(df)) * 100
            print(f"  - Missing Values (NaN): {nan_count} ({nan_percentage:.2f}%)")
        else:
            print("  - No Missing Values (NaN)")

        # Check data type
        dtype = df[col].dtype
        print(f"  - Data Type: {dtype}")

        # For object (string) or categorical columns, show unique values and their counts
        if dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
            unique_values = df[col].nunique()
            if unique_values < 20 and unique_values > 0:  # Limit for readability
                print(f"  - Unique Values ({unique_values}):")
                print(df[col].value_counts(dropna=False))
            elif unique_values > 20:
                print(f"  - Unique Values: {unique_values} (too many to list all, showing top 5)")
                print(df[col].value_counts(dropna=False).head())
            else: # unique_values == 0 (all NaN in an object column)
                print("  - No unique non-NaN values.")
        
        # For numeric columns, show basic stats if no NaNs, or min/max with NaNs
        elif pd.api.types.is_numeric_dtype(df[col]):
            if nan_count == 0:
                print(f"  - Min: {df[col].min()}, Max: {df[col].max()}")
                print(f"  - Mean: {df[col].mean():.2f}, Median: {df[col].median():.2f}")
            else:
                print(f"  - Min (non-NaN): {df[col].min()}, Max (non-NaN): {df[col].max()}")
        
        print("-" * (len(col) + 14) + "") # Separator for readability

In [23]:
check_dataframe_inconsistencies(df)

--- Checking DataFrame for Inconsistencies ---
Total rows: 142079
Total columns: 110
--- Column: 'sofifa_id' ---
  - No Missing Values (NaN)
  - Data Type: int64
  - Min: 2, Max: 264640
  - Mean: 211625.24, Median: 216393.00
-----------------------
--- Column: 'player_url' ---
  - No Missing Values (NaN)
  - Data Type: str
------------------------
--- Column: 'short_name' ---
  - No Missing Values (NaN)
  - Data Type: str
------------------------
--- Column: 'long_name' ---
  - No Missing Values (NaN)
  - Data Type: str
-----------------------
--- Column: 'player_positions' ---
  - No Missing Values (NaN)
  - Data Type: str
------------------------------
--- Column: 'overall' ---
  - No Missing Values (NaN)
  - Data Type: int64
  - Min: 40, Max: 94
  - Mean: 65.71, Median: 66.00
---------------------
--- Column: 'potential' ---
  - No Missing Values (NaN)
  - Data Type: int64
  - Min: 40, Max: 95
  - Mean: 70.73, Median: 70.00
-----------------------
--- Column: 'value_eur' ---
  - Mis

C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\3316436121.py:32: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\3316436121.py:32: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\3316436121.py:32: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\3316436121.py:32: Pandas4Warning: is_categorical_dtype is deprecated and will be re

  - Mean: 48.75, Median: 50.00
---------------------------------
--- Column: 'mentality_composure' ---
  - Missing Values (NaN): 31778 (22.37%)
  - Data Type: float64
  - Min (non-NaN): 3.0, Max (non-NaN): 96.0
---------------------------------
--- Column: 'defending_marking_awareness' ---
  - No Missing Values (NaN)
  - Data Type: int64
  - Min: 1, Max: 94
  - Mean: 45.66, Median: 50.00
-----------------------------------------
--- Column: 'defending_standing_tackle' ---
  - No Missing Values (NaN)
  - Data Type: int64
  - Min: 2, Max: 94
  - Mean: 47.61, Median: 54.00
---------------------------------------
--- Column: 'defending_sliding_tackle' ---
  - No Missing Values (NaN)
  - Data Type: int64
  - Min: 3, Max: 95
  - Mean: 45.65, Median: 52.00
--------------------------------------
--- Column: 'goalkeeping_diving' ---
  - No Missing Values (NaN)
  - Data Type: int64
  - Min: 1, Max: 91
  - Mean: 16.54, Median: 11.00
--------------------------------
--- Column: 'goalkeeping_handli

C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\3316436121.py:32: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\3316436121.py:32: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\3316436121.py:32: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
C:\Users\joshua.nghe\AppData\Local\Temp\ipykernel_14236\3316436121.py:32: Pandas4Warning: is_categorical_dtype is deprecated and will be re

In [24]:
def remove_goalkeepers(df):
    """
    Removes goalkeeper entries from the DataFrame based on the 'player_positions' column.

    Args:
        df (pd.DataFrame): The input DataFrame containing player data.

    Returns:
        pd.DataFrame: A new DataFrame with goalkeeper entries removed.
    """
    initial_rows = len(df)
    
    # Identify rows where 'player_positions' contains 'GK'
    # We use .str.contains() with regex=False for simple substring matching
    # and case=False to handle potential 'gk' or 'GK' variations
    # na=False ensures that NaN values in 'player_positions' do not raise an error
    goalkeeper_mask = df['player_positions'].str.contains('GK', na=False, case=False)
    
    # Filter out goalkeepers
    field_players_df = df[~goalkeeper_mask].copy()
    
    removed_rows = initial_rows - len(field_players_df)
    
    print(f"Successfully identified and removed {removed_rows} goalkeeper entries.")
    print(f"DataFrame shape before removal: {df.shape}")
    print(f"DataFrame shape after removal: {field_players_df.shape}")
    
    return field_players_df



In [25]:
print("--- Original DataFrame Info ---")
print(f"Number of rows: {len(df)}")
# Check initial distribution of GKs
print("Initial player_positions counts (top 5):")
print(df['player_positions'].value_counts().head())

# Remove goalkeepers
df_field_players = remove_goalkeepers(df.copy()) # Use a copy to not modify original df

print("--- DataFrame After Goalkeeper Removal ---")
print(f"Number of rows: {len(df_field_players)}")
# Verify that 'GK' is no longer a primary position
print("Player_positions counts after removal (top 5):")
print(df_field_players['player_positions'].value_counts().head())

--- Original DataFrame Info ---
Number of rows: 142079
Initial player_positions counts (top 5):
player_positions
CB    17551
GK    15791
ST    14372
CM     6311
LB     5576
Name: count, dtype: int64
Successfully identified and removed 15792 goalkeeper entries.
DataFrame shape before removal: (142079, 110)
DataFrame shape after removal: (126287, 110)
--- DataFrame After Goalkeeper Removal ---
Number of rows: 126287
Player_positions counts after removal (top 5):
player_positions
CB         17551
ST         14372
CM          6311
LB          5576
CDM, CM     5516
Name: count, dtype: int64
